# Activation Attention — Colab 容量扫描（E8 / E16 / moa4）

**用途**：在 T4 GPU 上跑本地跑不动的容量点（每个实验约 7~9 小时，1B tokens）。

**三步上手**：
1. 菜单「代码执行程序 → 更改代码执行程序类型」→ 选 **T4 GPU** → 保存
2. 从上到下依次运行下面所有 cell（第 4 格开始训练后，Colab 标签页保持打开即可，断了也能续）
3. 训练产物自动存 Google Drive（第 7 格），随时可断线重连

**要跑哪个实验**：修改「训练启动」格顶部的 `RUN_TAG` 配置区。建议顺序：
| RUN_TAG | 说明 | 预计时长 (T4) |
|---|---|---|
| `aa8_s42` / `aa8_s123` | AA 共享KV E=8 | ~6-7h |
| `aa16_s42` / `aa16_s123` | AA 共享KV E=16 | ~8-9h |
| `moa4_s42` / `moa4_s123` | 独立KV对照 | ~6h |

*(本地已跑：dense×2 与 aa{2,4}×2；本地 E∈{1,2,4} 结果与本表合并出 RQ1 曲线)*

In [ ]:
# 1. 确认 GPU（应为 Tesla T4，SM 7.5 → 自动走 fp16+GradScaler）
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| capability", torch.cuda.get_device_capability(0))

In [ ]:
# 2. 克隆框架 + 依赖（torch Colab 已自带，无需重装）
REPO_URL = "https://github.com/klxz111/-trainable-activate-attention-.git"
%cd /content
!rm -rf aa
!git clone {REPO_URL} aa
%cd aa
!pip install -q tiktoken datasets
!python -c "import sys; sys.path.insert(0,'.'); from aa.config import AAConfig; print('框架导入 OK')" 

In [ ]:
# 3. 下载数据（OpenWebText 10亿 token，约 5-10 分钟；存 /content/data，重克隆代码不丢）
import os
if os.path.exists("/content/data/owt1b/train.bin"):
    print("数据已存在，跳过下载")
else:
    %cd /content/aa
    !python scripts/prepare_data.py --dataset openwebtext --out /content/data/owt1b --max-tokens 1000000000
!ls -la /content/data/owt1b/

In [ ]:
# 4. 训练启动 —— 改 RUN_TAG 后运行本格（Popen 独立会话，cell 结束不杀进程）
RUN_TAG = "aa8_s42"   # 可选: aa8_s42 / aa8_s123 / aa16_s42 / aa16_s123 / moa4_s42 / moa4_s123

CFG_BASE = '{"n_layer":8,"n_embd":512,"n_head":8,"block_size":1024'
EXPS = {
    "aa8_s42":   ('"n_paths":8,"n_active":1,"shared_kv":true,"router_mode":"ste"',  "42"),
    "aa8_s123":  ('"n_paths":8,"n_active":1,"shared_kv":true,"router_mode":"ste"',  "123"),
    "aa16_s42":  ('"n_paths":16,"n_active":1,"shared_kv":true,"router_mode":"ste"', "42"),
    "aa16_s123": ('"n_paths":16,"n_active":1,"shared_kv":true,"router_mode":"ste"', "123"),
    "moa4_s42":  ('"n_paths":4,"n_active":1,"shared_kv":false,"router_mode":"ste"', "42"),
    "moa4_s123": ('"n_paths":4,"n_active":1,"shared_kv":false,"router_mode":"ste"', "123"),
}
extra, seed = EXPS[RUN_TAG]
ckpt_flag = "true" if RUN_TAG.startswith("aa16") else "false"
STEPS = 15300  # 1B tokens；怕断线可改 12000（0.8B，曲线仍可比）

import subprocess, time
ckpt = f"/content/aa/runs/sweep_{RUN_TAG}/ckpt_last.pt"
resume = f"--resume {ckpt}" if __import__('os').path.exists(ckpt) else ""
if resume: print(">>> 检测到断点，自动续训")

cmd = (
    "python -m aa.train --compile "
    "--data /content/data/owt1b --steps {st} "
    "--micro-batch 16 --accum 4 --eval-every 500 --save-every 500 --seed {sd} "
    "--config-json '{cb},\"grad_checkpoint\":{ck},{ex}}}' {rs} "
    "--out /content/aa/runs/sweep_{rt} > /content/aa/train_{rt}.log 2>&1"
).format(st=STEPS, sd=seed, cb=CFG_BASE, ck=ckpt_flag, ex=extra, rs=resume, rt=RUN_TAG)

log = open(f"/content/aa/train_{RUN_TAG}.log", "w")
proc = subprocess.Popen(cmd, shell=True, cwd="/content/aa", stdout=log,
                        stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(25)
alive = proc.poll() is None
print(f"进程存活: {alive}" + ("" if alive else " —— 启动失败，看下方日志排错"))
!tail -5 /content/aa/train_{RUN_TAG}.log
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

In [ ]:
# 5. 进度监控 —— 随时重跑本格
!tail -4 /content/aa/train_{RUN_TAG}.log
!echo "--- metrics ---"
!tail -2 /content/aa/runs/sweep_{RUN_TAG}/metrics.jsonl 2>/dev/null || echo "第一个 eval 在 step 500 后出现"
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

In [ ]:
# 6. （仅当会话断线重连后使用）恢复训练
#    重连后：先重跑第 2 格（重新 clone）+ 第 3 格（数据），然后：
#    A) 若之前把 ckpt 存到了 Drive（第 7 格）：先从 Drive 拷回
#    B) 再重跑第 4 格 —— 它检测到 ckpt_last.pt 会自动 --resume
print("断线恢复流程：第2格 → 第3格 → 第7格(若已备份) → 第4格（自动 resume）")

In [ ]:
# 7. 备份到 Google Drive（训练中随时可跑；建议每次看完进度顺手跑一次）
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
DEST = f"/content/drive/MyDrive/activation_attention_runs/sweep_{RUN_TAG}"
os.makedirs(DEST, exist_ok=True)
for f in ["metrics.jsonl", "ckpt_last.pt"]:
    src = f"runs/sweep_{RUN_TAG}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{DEST}/{f}")
        print(f"已备份 {f} -> Drive")
!ls -la {DEST}

## 跑完之后

- `train_{RUN_TAG}.log` 出现 `saved -> runs/sweep_{RUN_TAG}/ckpt.pt` 即完成
- 用第 7 格做最终备份，然后**下载 `metrics.jsonl`**（或整个 Drive 目录）回本地
- 本地合并：把 6 个 `metrics.jsonl` 放进 `runs/sweep_*/` 对应目录后运行
  `python scripts/plot_sweep.py --runs 'runs/sweep_*'` 出 RQ1 对比表
- 判读框架见 `plans/2026-10-16-stage-a-capacity-scaling.md` Task 10 与 `docs/stage-a-status.md`